# DD Startup Analysis - Parallel Coordinates Plot

This notebook reads HDF5 results from DD startup simulations and creates an interactive parallel coordinates plot showing the relationship between input parameters and economic outcomes.

## Configuration Options

Set your analysis parameters here before running the notebook:

In [ ]:
# =============================================================================
# CONFIGURATION OPTIONS - Set these before running the analysis
# =============================================================================

# File selection - Choose which HDF5 file to analyze
SELECTED_FILE = "dd_startup_results_20250916_161036.h5"  # Set to None for automatic selection, or specify filename
# Examples:
# SELECTED_FILE = 'dd_startup_results_20250916_120903.h5'
# default to newest

# Target variable selection - Choose which output metric to visualize
TARGET_VARIABLE = 'Dollar_Lost'    # Primary target variable for coloring
# Alternative target variables you can use:
# TARGET_VARIABLE = 't_startup'           # Startup time [s]
# TARGET_VARIABLE = 'Q_DD_total'          # Q factor for DD phase
# TARGET_VARIABLE = 'Q_DT_full_total'     # Q factor for DT phase
# TARGET_VARIABLE = 'P_fusion_DD_avg'     # Average DD fusion power [MW]
# TARGET_VARIABLE = 'P_e_net_DD_avg'      # Average DD net electric power [MW]
# TARGET_VARIABLE = 'E_fusion_total_DD'   # Total DD fusion energy [MJ]
# TARGET_VARIABLE = 'E_e_net_DD'          # Net DD electric energy [MJ]
# TARGET_VARIABLE = 'n_T_final'           # Final tritium density [m⁻³]

# Filter (sets the maximum value for color scaling)
FILTER = 1e6        # Set to threshold value or None

# Color and styling options
N_COLOR_CHUNKS = 6           # Number of discrete color levels (None sets to gradient)

## 1. Import Required Libraries

In [ ]:
import h5py
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from pathlib import Path

# Set plotly to work in notebooks
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)


## 2. Read H5 File Data

In [ ]:
# Define available files and apply configuration-based selection
outputs_dir = Path("outputs")
h5_files = list(outputs_dir.glob("*.h5"))

print("Available HDF5 files:")
for i, file in enumerate(h5_files):
    with h5py.File(file, 'r') as f:
        n_combinations = f.attrs.get('total_combinations', 'unknown')
        test_mode = f.attrs.get('test_mode', 'unknown')
        file_size = file.stat().st_size / 1024**2  # MB
    print(f"{i+1}. {file.name} - {n_combinations} combinations ({test_mode}) - {file_size:.1f} MB")

# File selection based on configuration
if SELECTED_FILE is not None:
    # Use explicitly specified file
    selected_file_path = outputs_dir / SELECTED_FILE
    if selected_file_path.exists():
        selected_file = selected_file_path
        print(f"\n🎯 Using explicitly selected file: {SELECTED_FILE}")
    else:
        print(f"\n❌ Specified file '{SELECTED_FILE}' not found!")
        print(f"Available files: {[f.name for f in h5_files]}")
        raise FileNotFoundError(f"File '{SELECTED_FILE}' not found in outputs directory")
else:
    # Automatic selection - always use most recent file
    if h5_files:
        selected_file = max(h5_files, key=lambda f: f.stat().st_mtime)
        print(f"\n📊 Using most recent file: {selected_file.name}")
    else:
        raise FileNotFoundError("No HDF5 files found in outputs directory")

print(f"✅ Selected file: {selected_file.name}")

# Load data from HDF5 file with better error handling
with h5py.File(selected_file, 'r') as f:
    print(f"\nFile metadata:")
    for key, value in f.attrs.items():
        print(f"  {key}: {value}")
    
    print(f"\nAvailable datasets:")
    main_datasets = []
    other_items = []
    
    for key in f.keys():
        if isinstance(f[key], h5py.Dataset):
            shape = f[key].shape
            dtype = f[key].dtype
            print(f"  {key}: shape {shape}, dtype {dtype}")
            # Only include 1D datasets for main data
            if len(shape) == 1:
                main_datasets.append(key)
            else:
                other_items.append(key)
        else:
            print(f"  {key}: {type(f[key])} (group)")
            other_items.append(key)
    
    print(f"\nLoading main datasets: {len(main_datasets)} items")
    
    # Load only 1D datasets that are likely to be main result data
    data = {}
    expected_length = None
    
    for key in main_datasets:
        try:
            dataset_array = f[key][:]
            if expected_length is None:
                expected_length = len(dataset_array)
                print(f"Setting expected length from '{key}': {expected_length}")
            
            if len(dataset_array) == expected_length:
                data[key] = dataset_array
                print(f"  ✅ Loaded {key}: {len(dataset_array)} elements")
            else:
                print(f"  ⚠️  Skipped {key}: length mismatch ({len(dataset_array)} vs {expected_length})")
                
        except Exception as e:
            print(f"  ❌ Error loading {key}: {e}")
    
    # Handle groups (like parameter_fields) separately if needed
    if 'parameter_fields' in f:
        print(f"\nFound parameter_fields group:")
        param_group = f['parameter_fields']
        for subkey in param_group.keys():
            try:
                param_data = param_group[subkey][:]
                print(f"  {subkey}: {param_data.shape} - {param_data}")
            except Exception as e:
                print(f"  Error reading {subkey}: {e}")

print(f"\nSuccessfully loaded {len(data)} datasets")
if len(data) > 0:
    # Use the first dataset to determine the number of combinations
    first_key = list(data.keys())[0]
    n_combinations = len(data[first_key])
    print(f"Number of combinations: {n_combinations}")
    
    # Show a sample of what was loaded
    print(f"\nLoaded datasets:")
    for key, value in data.items():
        print(f"  {key}: length {len(value)}, dtype {value.dtype}")
        if len(value) > 0:
            print(f"    Sample: min={np.min(value):.3e}, max={np.max(value):.3e}")
else:
    print("❌ No valid datasets loaded!")
    raise ValueError("No data could be loaded from the HDF5 file")

## 3. Data Preprocessing and Filtering

In [ ]:
# Convert to pandas DataFrame for easier manipulation
# First, check the data structure and handle different array lengths
print("Checking data structure:")
for key, value in data.items():
    if isinstance(value, np.ndarray):
        print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
    else:
        print(f"  {key}: type {type(value)}")

# Filter out datasets that don't match the main data length
# Use linear_index as reference for the correct length
if 'linear_index' in data:
    expected_length = len(data['linear_index'])
    print(f"\nExpected data length (from linear_index): {expected_length}")
    
    # Filter data to only include arrays with the correct length
    filtered_data = {}
    for key, value in data.items():
        if isinstance(value, np.ndarray) and len(value) == expected_length:
            filtered_data[key] = value
        elif not isinstance(value, np.ndarray):
            print(f"Skipping non-array data: {key}")
        else:
            print(f"Skipping mismatched length data: {key} (length {len(value)})")
    
    print(f"\nFiltered data contains {len(filtered_data)} arrays with correct length")
    data_for_df = filtered_data
else:
    print("Warning: 'linear_index' not found. Using all data as-is.")
    data_for_df = data

# Create DataFrame from filtered data
try:
    df = pd.DataFrame(data_for_df)
    print(f"✅ Successfully created DataFrame with shape: {df.shape}")
except Exception as e:
    print(f"❌ Error creating DataFrame: {e}")
    # If still failing, let's try a more careful approach
    print("Attempting manual DataFrame creation...")
    
    # Find the most common array length
    lengths = []
    for key, value in data_for_df.items():
        if isinstance(value, np.ndarray):
            lengths.append(len(value))
    
    if lengths:
        most_common_length = max(set(lengths), key=lengths.count)
        print(f"Most common array length: {most_common_length}")
        
        # Use only arrays with the most common length
        final_data = {}
        for key, value in data_for_df.items():
            if isinstance(value, np.ndarray) and len(value) == most_common_length:
                final_data[key] = value
        
        df = pd.DataFrame(final_data)
        print(f"✅ Created DataFrame with shape: {df.shape}")
    else:
        raise ValueError("No valid arrays found for DataFrame creation")

print(f"Columns: {list(df.columns)}")

# Define input parameters (excluding calculated variables)
input_parameters = [
    'V_plasma',           # Plasma volume
    'T_i',               # Ion temperature
    'n_tot',             # Total density
    'tau_p_T',           # Tritium particle confinement time
    'tau_p_He3',         # He3 particle confinement time
    'P_aux',             # Auxiliary power (DD phase)
    'P_lost_rad',        # Radiated power loss (DD phase)
    'P_aux_all_DT',      # Auxiliary power (DT phase)
    'P_lost_rad_all_DT', # Radiated power loss (DT phase)
    'TBR_DT',            # Tritium breeding ratio (DT)
    'TBR_DDn',           # Tritium breeding ratio (DD neutron)
    'tau_ifc',           # In-facility cycle time
    'tau_ofc',           # Out-of-facility cycle time
    'eta_th',            # Thermal efficiency
    'plant_avail',       # Plant availability
    'Cost_per_kWh'       # Cost per kWh
]

# Target variable for coloring (use the configured target)
target_variable = TARGET_VARIABLE

# Variables to exclude from input parameters (but can be used as targets)
excluded_variables = [
    'linear_index',
    'injection_rate_max', 'sigmav_DT', 'sigmav_DD_p', 'sigmav_DD_n',  # Calculated from inputs
    'P_DT', 'P_DDn', 'P_DDp', 'P_DT_full',                          # ODE results
    'sol_success'                                                    # Solution status
]

# Available output metrics that can be used as target variables:
available_target_variables = [
    'Dollar_Lost',           # Economic metric [$]
    't_startup',            # Startup time [s]
    'Q_DD_total',           # Q factor for DD phase [-]
    'Q_DT_full_total',      # Q factor for DT phase [-]
    'P_fusion_DD_avg',      # Average DD fusion power [MW]
    'P_e_net_DD_avg',       # Average DD net electric power [MW]
    'P_e_net_DT_full_avg',  # Average DT net electric power [MW]
    'E_fusion_total_DD',    # Total DD fusion energy [MJ]
    'E_fusion_DT_full',     # Total DT fusion energy [MJ]
    'E_e_net_DD',           # Net DD electric energy [MJ]
    'E_e_net_DT_full',      # Net DT electric energy [MJ]
    'E_lost',               # Lost energy [MJ]
    'n_T_final'             # Final tritium density [m⁻³]
]

# Check which input parameters are actually available in the DataFrame
available_input_params = [param for param in input_parameters if param in df.columns]
missing_input_params = [param for param in input_parameters if param not in df.columns]

print(f"\nInput parameters analysis:")
print(f"  Available: {len(available_input_params)} / {len(input_parameters)}")
for param in available_input_params:
    print(f"    ✅ {param}")

if missing_input_params:
    print(f"  Missing: {len(missing_input_params)}")
    for param in missing_input_params:
        print(f"    ❌ {param}")

# Update input_parameters to only include available ones
input_parameters = available_input_params

# Check target variable
if target_variable not in df.columns:
    print(f"\n❌ Target variable '{target_variable}' not found in data!")
    print(f"Available columns: {list(df.columns)}")
    
    # Show available target options
    available_targets = [col for col in available_target_variables if col in df.columns]
    if available_targets:
        print(f"\n🎯 Available target variables:")
        for target in available_targets:
            print(f"    ✅ {target}")
        
        # Use the first available target as fallback
        target_variable = available_targets[0]
        print(f"\n🔄 Using fallback target variable: {target_variable}")
    else:
        print(f"\n❌ No suitable target variables found!")
        raise ValueError("No suitable target variable found")
else:
    print(f"✅ Target variable '{target_variable}' found")

# Show target variable info
if target_variable in df.columns:
    values = df[target_variable]
    finite_vals = values[np.isfinite(values)]
    if len(finite_vals) > 0:
        print(f"📊 {target_variable} statistics:")
        print(f"   Range: {finite_vals.min():.2e} to {finite_vals.max():.2e}")
        print(f"   Mean: {finite_vals.mean():.2e}")
        print(f"   Median: {finite_vals.median():.2e}")

print(f"\nFinal configuration:")
print(f"  Input parameters: {len(input_parameters)}")
print(f"  Target variable: {target_variable}")
print(f"  DataFrame shape: {df.shape}")

# Filter out infinite and invalid values
print(f"\nFiltering data...")
print(f"Before filtering: {len(df)} rows")

# Keep only finite target variable values
finite_mask = np.isfinite(df[target_variable])
df_filtered = df[finite_mask].copy()

print(f"After filtering infinite {target_variable}: {len(df_filtered)} rows")

# =============================================================================
# ECONOMIC FILTER: Maximum Dollar Lost threshold
# =============================================================================
# Set maximum Dollar_Lost threshold (None = no filter)
# You can adjust this value or set to None to include all scenarios

max_dollar_lost = None  # Set to a value (e.g., 1e8) to filter, or None for no filter
# max_dollar_lost = 1e8  # Example: Only scenarios with Dollar_Lost <= $100M

if max_dollar_lost is not None:
    print(f"\n🎯 Applying economic filter: {target_variable} <= ${max_dollar_lost:.2e}")
    economic_mask = df_filtered[target_variable] <= max_dollar_lost
    df_filtered = df_filtered[economic_mask].copy()
    print(f"After economic filtering: {len(df_filtered)} rows")
    print(f"Economic viability rate: {len(df_filtered)/len(df[finite_mask])*100:.1f}% of successful scenarios")
else:
    print(f"\n⚠️  No economic filter applied - showing all scenarios")

print(f"Success rate: {len(df_filtered)/len(df)*100:.1f}%")

# Display target variable distribution for reference
if len(df_filtered) > 0:
    target_stats = df_filtered[target_variable].describe()
    print(f"\n{target_variable} distribution in filtered data:")
    print(f"  Count:  {target_stats['count']:.0f}")
    print(f"  Min:    ${target_stats['min']:.2e}")
    print(f"  25%:    ${target_stats['25%']:.2e}")
    print(f"  Median: ${target_stats['50%']:.2e}")
    print(f"  75%:    ${target_stats['75%']:.2e}")
    print(f"  Max:    ${target_stats['max']:.2e}")
else:
    print(f"\n❌ No data remaining after filtering!")

print(f"\nFinal dataframe for plotting: {df_filtered.shape}")
print(f"Input parameters: {len(input_parameters)}")
if len(df_filtered) > 0:
    print(f"Target variable range: {df_filtered[target_variable].min():.2e} to {df_filtered[target_variable].max():.2e}")

# Show filter status for user reference
if max_dollar_lost is not None:
    print(f"\n📊 Economic Filter Status: ACTIVE (Max {target_variable} = ${max_dollar_lost:.2e})")
    print(f"   To disable filter: Set max_dollar_lost = None")
    print(f"   To adjust filter: Set max_dollar_lost = your_desired_value")
else:
    print(f"\n📊 Economic Filter Status: DISABLED")
    print(f"   To enable filter: Set max_dollar_lost = your_desired_threshold (e.g., 1e8)")
    print(f"   Example thresholds: 1e7 ($10M), 1e8 ($100M), 1e9 ($1B)")

## 4. Identify Input Variables and Target Variable

In [ ]:
# Create readable labels for the plot
parameter_labels = {
    # Input parameters
    'V_plasma': 'Plasma Volume [m³]',
    'T_i': 'Ion Temperature [keV]',
    'n_tot': 'Total Density [m⁻³]',
    'tau_p_T': 'Tritium τp [s]',
    'tau_p_He3': 'He3 τp [s]',
    'P_aux': 'P_aux DD [MW]',
    'P_lost_rad': 'P_rad DD [MW]',
    'P_aux_all_DT': 'P_aux DT [MW]',
    'P_lost_rad_all_DT': 'P_rad DT [MW]',
    'TBR_DT': 'TBR DT [-]',
    'TBR_DDn': 'TBR DDn [-]',
    'tau_ifc': 'τ_ifc [h]',
    'tau_ofc': 'τ_ofc [h]',
    'eta_th': 'η_thermal [-]',
    'plant_avail': 'Plant Avail. [-]',
    'Cost_per_kWh': 'Cost [$/kWh]',
    
    # Output metrics (potential target variables)
    'Dollar_Lost': 'Dollar Lost [$]',
    't_startup': 'Startup Time [s]',
    'Q_DD_total': 'Q Factor DD [-]',
    'Q_DT_full_total': 'Q Factor DT [-]',
    'P_fusion_DD_avg': 'Avg DD Fusion Power [MW]',
    'P_e_net_DD_avg': 'Avg DD Net Power [MW]',
    'P_e_net_DT_full_avg': 'Avg DT Net Power [MW]',
    'E_fusion_total_DD': 'Total DD Fusion Energy [MJ]',
    'E_fusion_DT_full': 'Total DT Fusion Energy [MJ]',
    'E_e_net_DD': 'Net DD Electric Energy [MJ]',
    'E_e_net_DT_full': 'Net DT Electric Energy [MJ]',
    'E_lost': 'Lost Energy [MJ]',
    'n_T_final': 'Final Tritium Density [m⁻³]'
}

# Display parameter statistics
print("Parameter Statistics:")
print("=" * 80)
for param in input_parameters + [target_variable]:
    if param in df_filtered.columns:
        values = df_filtered[param]
        print(f"{parameter_labels.get(param, param):20s}: "
              f"min={values.min():8.2e}, max={values.max():8.2e}, "
              f"mean={values.mean():8.2e}, std={values.std():8.2e}")

# Check parameter ranges to ensure good visualization
print(f"\nParameter Ranges Check:")
for param in input_parameters:
    if param in df_filtered.columns:
        values = df_filtered[param]
        unique_vals = len(values.unique())
        range_ratio = (values.max() - values.min()) / values.std() if values.std() > 0 else 0
        print(f"{param:20s}: {unique_vals:3d} unique values, range/std = {range_ratio:.2f}")

# Create a subset for visualization if dataset is too large
max_points = 1000  # Limit for better performance
if len(df_filtered) > max_points:
    print(f"\nDataset has {len(df_filtered)} points. Sampling {max_points} for visualization...")
    df_plot = df_filtered.sample(n=max_points, random_state=42).copy()
else:
    df_plot = df_filtered.copy()

print(f"Using {len(df_plot)} points for visualization")

## 5. Create Parallel Coordinates Plot

In [ ]:
# =============================================================================
# MAIN PLOTTING SECTION - Single Parallel Coordinates Plot
# =============================================================================

def create_paracoords_plot(data_df, target_var, filter_value=None, n_chunks=None):
    """
    Create parallel coordinates plot with configurable target variable and filtering
    
    Parameters:
    - data_df: DataFrame with all data
    - target_var: Target variable for coloring
    - filter_value: Filter threshold (None for no filter)
    - n_chunks: Number of color chunks (None for gradient)
    """
    
    # Apply basic filtering for finite values
    df_filtered_plot = data_df[np.isfinite(data_df[target_var])].copy()
    
    # Apply threshold filter if specified
    if filter_value is not None:
        if target_var in ['t_startup', 'Dollar_Lost', 'E_lost']:
            # For these metrics, lower is better
            filter_mask = df_filtered_plot[target_var] <= filter_value
            filter_text = f"Filter: {target_var} ≤ {filter_value:.2e}"
        else:
            # For Q factors, powers, energies, higher is typically better
            filter_mask = df_filtered_plot[target_var] >= filter_value
            filter_text = f"Filter: {target_var} ≥ {filter_value:.2e}"
        
        df_filtered_plot = df_filtered_plot[filter_mask].copy()
    else:
        filter_text = "No Filter"
    
    if len(df_filtered_plot) == 0:
        print(f"❌ No scenarios meet the filter criteria!")
        print(f"   Filter: {filter_text}")
        print(f"   Try adjusting the threshold or set FILTER = None")
        return None
    
    # Sample data if too large
    max_points = 1000
    if len(df_filtered_plot) > max_points:
        df_plot_final = df_filtered_plot.sample(n=max_points, random_state=42).copy()
        sample_text = f"(sampled {max_points} points)"
    else:
        df_plot_final = df_filtered_plot.copy()
        sample_text = ""
    
    # Create discrete color chunks or use gradient
    use_discrete_colors = n_chunks is not None
    
    if use_discrete_colors:
        target_values = df_plot_final[target_var]
        
        # Define quantile-based chunks for better distribution
        quantiles = np.linspace(0, 1, n_chunks + 1)
        chunk_boundaries = target_values.quantile(quantiles).values
        
        # Create discrete color values
        color_values = np.zeros(len(target_values))
        
        # Define color scheme based on target variable
        if target_var in ['t_startup', 'Dollar_Lost', 'E_lost']:
            # For "lower is better" metrics: Green (best) → Red (worst)
            colors = ['#2E8B57', '#32CD32', '#FFD700', '#FF8C00', '#FF4500', '#8B0000'][:n_chunks]
        else:
            # For "higher is better" metrics: reverse the order
            colors = ['#8B0000', '#FF4500', '#FF8C00', '#FFD700', '#32CD32', '#2E8B57'][:n_chunks]
            colors = colors[::-1]  # Reverse so green is still "good"
        
        for i in range(n_chunks):
            if i == 0:
                mask = (target_values >= chunk_boundaries[i]) & (target_values <= chunk_boundaries[i+1])
            elif i == n_chunks - 1:
                mask = target_values > chunk_boundaries[i]
            else:
                mask = (target_values > chunk_boundaries[i]) & (target_values <= chunk_boundaries[i+1])
            
            color_values[mask] = i
        
        # Create custom discrete colorscale
        discrete_colorscale = []
        for i in range(n_chunks):
            discrete_colorscale.extend([
                [i/(n_chunks-1), colors[i]], 
                [i/(n_chunks-1), colors[i]]
            ])
        
        colorscale = discrete_colorscale
        color_data = color_values
        
    else:
        # Use continuous gradient
        if target_var in ['t_startup', 'Dollar_Lost', 'E_lost']:
            colorscale = 'RdYlGn_r'  # Red (bad) to Green (good)
        else:
            colorscale = 'RdYlGn'    # Red (bad) to Green (good), but reversed
        color_data = df_plot_final[target_var]
    
    # Create dimensions for parallel coordinates
    dimensions = []
    for param in input_parameters:
        if param in df_plot_final.columns:
            values = df_plot_final[param]
            dimensions.append(
                dict(
                    label=parameter_labels.get(param, param),
                    values=values,
                    range=[values.min(), values.max()]
                )
            )
    
    # Add target variable as last dimension
    target_values_final = df_plot_final[target_var]
    dimensions.append(
        dict(
            label=parameter_labels.get(target_var, target_var),
            values=target_values_final,
            range=[target_values_final.min(), target_values_final.max()]
        )
    )
    
    # Create the plot
    fig = go.Figure(data=
        go.Parcoords(
            line=dict(
                color=color_data,
                colorscale=colorscale,
                showscale=True,
                colorbar=dict(
                    title=dict(
                        text=parameter_labels.get(target_var, target_var),
                        font=dict(size=14)
                    ),
                    thickness=20,
                    len=0.8,
                    tickfont=dict(size=11)
                ),
                cmin=0 if use_discrete_colors else target_values_final.min(),
                cmax=n_chunks-1 if use_discrete_colors else target_values_final.max()
            ),
            dimensions=dimensions
        )
    )
    
    # Calculate statistics
    success_rate = len(df_filtered_plot) / len(data_df[np.isfinite(data_df[target_var])]) * 100
    
    # Update layout
    color_type = f"Discrete Colors ({n_chunks} levels)" if use_discrete_colors else "Gradient Colors"
    fig.update_layout(
        title=dict(
            text=f"DD Startup Parameter Sensitivity Analysis<br>"
                 f"<sub>{len(df_plot_final)} data points {sample_text} • "
                 f"Success rate: {success_rate:.1f}% • "
                 f"Target: {parameter_labels.get(target_var, target_var)} • "
                 f"{filter_text} • {color_type}</sub>",
            x=0.5,
            font=dict(size=18)
        ),
        font=dict(size=12),
        width=1400,
        height=700,
        margin=dict(l=100, r=120, t=120, b=100),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    return fig

# =============================================================================
# CREATE THE PLOT
# =============================================================================

print("🎨 Creating parallel coordinates plot...")
print(f"🎯 Target Variable: {TARGET_VARIABLE}")
print(f"📊 Filter: {FILTER}")
print(f"🌈 Color scheme: {'Discrete chunks (' + str(N_COLOR_CHUNKS) + ' levels)' if N_COLOR_CHUNKS else 'Continuous gradient'}")

# Create the plot
fig = create_paracoords_plot(
    df, 
    target_var=TARGET_VARIABLE,
    filter_value=FILTER,
    n_chunks=N_COLOR_CHUNKS
)

if fig is not None:
    fig.show()
    
    # Display summary statistics
    df_filtered = df[np.isfinite(df[TARGET_VARIABLE])].copy()
    if FILTER is not None:
        if TARGET_VARIABLE in ['t_startup', 'Dollar_Lost', 'E_lost']:
            df_filtered = df_filtered[df_filtered[TARGET_VARIABLE] <= FILTER]
        else:
            df_filtered = df_filtered[df_filtered[TARGET_VARIABLE] >= FILTER]
    
    print(f"\n📈 Analysis Summary:")
    print(f"   • Total scenarios analyzed: {len(df_filtered)}")
    
    if TARGET_VARIABLE == 'Dollar_Lost':
        print(f"   • {TARGET_VARIABLE} range: ${df_filtered[TARGET_VARIABLE].min():.2e} to ${df_filtered[TARGET_VARIABLE].max():.2e}")
        print(f"   • Median {TARGET_VARIABLE}: ${df_filtered[TARGET_VARIABLE].median():.2e}")
    else:
        print(f"   • {TARGET_VARIABLE} range: {df_filtered[TARGET_VARIABLE].min():.2e} to {df_filtered[TARGET_VARIABLE].max():.2e}")
        print(f"   • Median {TARGET_VARIABLE}: {df_filtered[TARGET_VARIABLE].median():.2e}")
    
    if N_COLOR_CHUNKS:
        print(f"\n🎯 How to interpret colors:")
        if TARGET_VARIABLE in ['t_startup', 'Dollar_Lost', 'E_lost']:
            print(f"   🟢 Green shades: Best performance (lower values)")
            print(f"   🟡 Yellow/Orange: Moderate performance")
            print(f"   🔴 Red shades: Poor performance (higher values)")
        else:
            print(f"   🟢 Green shades: Best performance (higher values)")
            print(f"   🟡 Yellow/Orange: Moderate performance")
            print(f"   🔴 Red shades: Poor performance (lower values)")
    
    print(f"\n💡 Interactive Features:")
    print(f"   • Click and drag on any axis to filter parameter ranges")
    print(f"   • Focus on green lines for optimal parameter combinations")
    print(f"   • Use the color bar to understand performance levels")

else:
    print("❌ Unable to create plot. Check your filter settings.")
    # Show some helpful information about the data
    if len(df) > 0 and TARGET_VARIABLE in df.columns:
        target_vals = df[TARGET_VARIABLE][np.isfinite(df[TARGET_VARIABLE])]
        min_val = target_vals.min()
        max_val = target_vals.max()
        median_val = target_vals.median()
        
        print(f"\n📊 Available {TARGET_VARIABLE} range:")
        if TARGET_VARIABLE == 'Dollar_Lost':
            print(f"   • Minimum: ${min_val:.2e}")
            print(f"   • Median: ${median_val:.2e}")
            print(f"   • Maximum: ${max_val:.2e}")
            if FILTER is not None:
                print(f"   • Current filter: ${FILTER:.2e}")
        else:
            print(f"   • Minimum: {min_val:.2e}")
            print(f"   • Median: {median_val:.2e}")
            print(f"   • Maximum: {max_val:.2e}")
            if FILTER is not None:
                print(f"   • Current filter: {FILTER:.2e}")

print(f"\n⚙️  Configuration Options:")
print(f"   • Change TARGET_VARIABLE to explore different metrics")
print(f"   • Adjust FILTER to focus on specific performance ranges")
print(f"   • Set N_COLOR_CHUNKS = None for gradient colors")
print(f"   • Modify N_COLOR_CHUNKS (4-8) for different discrete levels")


In [ ]:
# =============================================================================
# CHECK FOR SUSPICIOUS DOLLAR_LOST ≈ 0 COMBINATIONS
# =============================================================================

# Check for combinations that lead to Dollar_Lost ≈ 0 (within ±10)
if 'Dollar_Lost' in df.columns:
    print(f"\n" + "="*80)
    print("🔍 CHECKING FOR SUSPICIOUS DOLLAR_LOST ≈ 0 COMBINATIONS")
    print("="*80)
    
    # Find scenarios where Dollar_Lost is approximately zero (within ±10)
    near_zero_mask = (np.abs(df['Dollar_Lost']) <= 10) & np.isfinite(df['Dollar_Lost'])
    near_zero_scenarios = df[near_zero_mask].copy()
    
    if len(near_zero_scenarios) > 0:
        print(f"⚠️  WARNING: Found {len(near_zero_scenarios)} scenarios with Dollar_Lost ≈ 0 (±10)")
        print(f"   This is unexpected and may indicate:")
        print(f"   • Calculation errors")
        print(f"   • Unrealistic parameter combinations")
        print(f"   • Model boundary conditions")
        
        print(f"\n📋 Input Parameter Combinations Leading to Dollar_Lost ≈ 0:")
        print("-" * 80)
        
        # Display the input parameters for these suspicious cases
        display_params = [param for param in input_parameters if param in near_zero_scenarios.columns]
        
        for idx, (row_idx, row) in enumerate(near_zero_scenarios.iterrows(), 1):
            print(f"\n🔸 Scenario {idx} (Dollar_Lost = ${row['Dollar_Lost']:.2f}):")
            
            # Group parameters by type for better readability
            plasma_params = ['V_plasma', 'T_i', 'n_tot']
            confinement_params = ['tau_p_T', 'tau_p_He3']
            power_params = ['P_aux', 'P_lost_rad', 'P_aux_all_DT', 'P_lost_rad_all_DT']
            breeding_params = ['TBR_DT', 'TBR_DDn']
            operational_params = ['tau_ifc', 'tau_ofc', 'eta_th', 'plant_avail']
            economic_params = ['Cost_per_kWh']
            
            param_groups = [
                ("Plasma", plasma_params),
                ("Confinement", confinement_params),
                ("Power", power_params),
                ("Breeding", breeding_params),
                ("Operational", operational_params),
                ("Economic", economic_params)
            ]
            
            for group_name, group_params in param_groups:
                group_values = []
                for param in group_params:
                    if param in display_params:
                        label = parameter_labels.get(param, param)
                        value = row[param]
                        group_values.append(f"{param}={value:.3e}")
                
                if group_values:
                    print(f"     {group_name:12s}: {', '.join(group_values)}")
            
            # Show other relevant outputs if available
            other_outputs = ['t_startup', 'Q_DD_total', 'Q_DT_full_total', 'P_fusion_DD_avg', 'E_e_net_DD']
            output_values = []
            for output in other_outputs:
                if output in row.index and np.isfinite(row[output]):
                    output_values.append(f"{output}={row[output]:.3e}")
            
            if output_values:
                print(f"     {'Outputs':12s}: {', '.join(output_values)}")
        
        print(f"\n💡 Recommendations:")
        print(f"   • Review these parameter combinations for physical plausibility")
        print(f"   • Check calculation logic for Dollar_Lost computation")
        print(f"   • Verify boundary conditions and constraints")
        print(f"   • Consider excluding these scenarios from analysis if deemed unphysical")
        
    else:
        print(f"✅ No scenarios found with Dollar_Lost ≈ 0 (±10)")
        print(f"   All scenarios have realistic Dollar_Lost values")
        print(f"   Dollar_Lost range: ${df['Dollar_Lost'][np.isfinite(df['Dollar_Lost'])].min():.2e} to ${df['Dollar_Lost'][np.isfinite(df['Dollar_Lost'])].max():.2e}")

else:
    print(f"\n⚠️  Dollar_Lost column not found in data - skipping suspicious value check")
    
print(f"\n" + "="*80)